In [ ]:
# Feature-extraction kernel, round 2 (no submission; internet ON). Round 1: frozen DeBERTa-v3-large L24+L6 mean
# pool = 0.445 CV and +0.0061 held out on the v7 set (18/20). Round 2 widens that family: every upper DeBERTa-v3-large
# layer with mean AND style-token pooling, DeBERTa-v2 xlarge/xxlarge, ELECTRA-large discriminator, frozen ModernBERT-large.
# Also saves DeBERTa-v3-large weights + tokenizer (deberta-v3-large/) so the offline submission notebook can attach
# this kernel's output.
ENCODERS = [  # (tag, HF id, layers, dtype)
    ('debv3l2', 'microsoft/deberta-v3-large', [6, 12, 16, 18, 20, 21, 22, 23, 24], 'fp32'),
    ('debv2xl', 'microsoft/deberta-v2-xlarge', [6, 12, 18, 21, 24], 'fp32'),
    ('debv2xxl', 'microsoft/deberta-v2-xxlarge', [12, 24, 36, 42, 48], 'fp32'),
    ('electl', 'google/electra-large-discriminator', [6, 12, 18, 21, 24], 'fp32'),
    ('mbertl', 'answerdotai/ModernBERT-large', [7, 14, 21, 25, 28], 'fp32'),
]
SAVE = 'microsoft/deberta-v3-large'
BS, MAX_LEN = 16, 512

import gc
import pathlib
import re
import time

import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from transformers import AutoModel, AutoTokenizer

T0 = time.time()
DATA_DIR = next(p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'),
                            pathlib.Path('/kaggle/input/name-that-llm'), pathlib.Path('data')]
                if (p / 'train.csv').exists())
train = pd.read_csv(DATA_DIR / 'train.csv')
TR = train['text'].fillna('').astype(str).tolist()
order = np.argsort([len(t) for t in TR])
dev = 'cuda'
FUNC = frozenset(ENGLISH_STOP_WORDS)
for tag, hf, layers, dt in ENCODERS:
    try:
        t0 = time.time()
        tok = AutoTokenizer.from_pretrained(hf)
        model = AutoModel.from_pretrained(hf, dtype=torch.float32).to(dev).eval()
        if hf == SAVE:
            model.save_pretrained('deberta-v3-large')
            tok.save_pretrained('deberta-v3-large')
        pieces = [tok.decode([i]) for i in range(len(tok))]
        smask = torch.tensor([(p.strip().lower() in FUNC) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces]).to(dev)
        H = model.config.hidden_size
        emb = np.zeros((len(TR), len(layers), H), dtype=np.float32)
        emb_s = np.zeros_like(emb)
        with torch.no_grad():
            for b0 in range(0, len(TR), BS):
                idx = order[b0:b0 + BS]
                enc = tok([TR[i] for i in idx], padding=True, truncation=True, max_length=MAX_LEN,
                          return_tensors='pt').to(dev)
                hs = model(**enc, output_hidden_states=True).hidden_states
                m = enc['attention_mask'].unsqueeze(-1).float()
                ms = m * smask[enc['input_ids']].unsqueeze(-1).float()
                for li, L in enumerate(layers):
                    h = hs[L].float()
                    emb[idx, li] = ((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
                    emb_s[idx, li] = ((h * ms).sum(1) / ms.sum(1).clamp(min=1)).cpu().numpy()
                del hs, enc
        print(tag, hf, 'layers', layers, 'of', model.config.num_hidden_layers, 'hidden', H, 'finite',
              bool(np.isfinite(emb).all() and np.isfinite(emb_s).all()), f'{time.time() - t0:.0f}s', flush=True)
        np.savez(f'enc_{tag}_train.npz', emb=emb.astype(np.float16), emb_style=emb_s.astype(np.float16),
                 layers=np.array(layers), ids=train['id'].values)
        del model, tok
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print('FAILED', tag, hf, repr(e)[:300], flush=True)
print('total', f'{time.time() - T0:.0f}s', flush=True)
pd.DataFrame({'id': [], 'model': []}).to_parquet('submission.parquet')   # placeholder (feature-extraction notebook)
